<a href="https://colab.research.google.com/github/LoneWolf206/sentiment-analyzer/blob/main/sentiment_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
files.upload()  # upload kaggle.json again

In [2]:
!mkdir ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d snap/amazon-fine-food-reviews
!unzip amazon-fine-food-reviews.zip

Dataset URL: https://www.kaggle.com/datasets/snap/amazon-fine-food-reviews
License(s): CC0-1.0
100% 242M/242M [00:15<00:00, 15.9MB/s]

Archive:  amazon-fine-food-reviews.zip
  inflating: Reviews.csv             
  inflating: database.sqlite         
  inflating: hashes.txt              


In [3]:
!unzip amazon-fine-food-reviews.zip

Archive:  amazon-fine-food-reviews.zip
replace Reviews.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace database.sqlite? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace hashes.txt? [y]es, [n]o, [A]ll, [N]one, [r]ename: n


In [4]:
import pandas as pd
df = pd.read_csv('Reviews.csv')
print(df.shape)
print(df.head())

(568454, 10)
   Id   ProductId          UserId                      ProfileName  \
0   1  B001E4KFG0  A3SGXH7AUHU8GW                       delmartian   
1   2  B00813GRG4  A1D87F6ZCVE5NK                           dll pa   
2   3  B000LQOCH0   ABXLMWJIXXAIN  Natalia Corres "Natalia Corres"   
3   4  B000UA0QIQ  A395BORC6FGVXV                             Karl   
4   5  B006K2ZZ7K  A1UQRSCLF8GW1T    Michael D. Bigham "M. Wassir"   

   HelpfulnessNumerator  HelpfulnessDenominator  Score        Time  \
0                     1                       1      5  1303862400   
1                     0                       0      1  1346976000   
2                     1                       1      4  1219017600   
3                     3                       3      2  1307923200   
4                     0                       0      5  1350777600   

                 Summary                                               Text  
0  Good Quality Dog Food  I have bought several of the Vitality can

In [5]:
print(df['Score'].value_counts())
print(df['Text'].isnull().sum())


Score
5    363122
4     80655
1     52268
3     42640
2     29769
Name: count, dtype: int64
0


In [6]:
print(df.describe())
print(df.info())

                  Id  HelpfulnessNumerator  HelpfulnessDenominator  \
count  568454.000000         568454.000000            568454.00000   
mean   284227.500000              1.743817                 2.22881   
std    164098.679298              7.636513                 8.28974   
min         1.000000              0.000000                 0.00000   
25%    142114.250000              0.000000                 0.00000   
50%    284227.500000              0.000000                 1.00000   
75%    426340.750000              2.000000                 2.00000   
max    568454.000000            866.000000               923.00000   

               Score          Time  
count  568454.000000  5.684540e+05  
mean        4.183199  1.296257e+09  
std         1.310436  4.804331e+07  
min         1.000000  9.393408e+08  
25%         4.000000  1.271290e+09  
50%         5.000000  1.311120e+09  
75%         5.000000  1.332720e+09  
max         5.000000  1.351210e+09  
<class 'pandas.core.frame.DataFrame'

In [7]:
# Convert scores to sentiment labels
def get_sentiment(score):
    if score <= 2:
        return 'negative'
    elif score == 3:
        return 'neutral'
    else:
        return 'positive'

df['sentiment'] = df['Score'].apply(get_sentiment)
print(df['sentiment'].value_counts())

sentiment
positive    443777
negative     82037
neutral      42640
Name: count, dtype: int64


In [8]:

# Sample equal numbers from each class
min_count = df['sentiment'].value_counts().min()
df_balanced = df.groupby('sentiment').sample(n=min_count, random_state=42)

print(df_balanced['sentiment'].value_counts())
print(df_balanced.shape)

sentiment
negative    42640
neutral     42640
positive    42640
Name: count, dtype: int64
(127920, 11)


In [9]:
import re

def clean_text(text):
    text = re.sub(r'<.*?>', '', text)        # remove HTML tags
    text = re.sub(r'[^a-zA-Z\s]', '', text)  # remove special characters
    text = text.lower().strip()               # lowercase
    return text

df_balanced['clean_text'] = df_balanced['Text'].apply(clean_text)
print(df_balanced['clean_text'].head())

525327    i have an absolute passion for deep dark hot c...
75760     this drink is so super energy its almost frigh...
468100    im sticking with what used to be carnation now...
71864     aspertame causes alot of problems including pr...
211592    i ordered these because my local pet store sto...
Name: clean_text, dtype: object


In [10]:
df_sample = df_balanced.sample(n=15000, random_state=42)
print(df_sample['sentiment'].value_counts())

sentiment
negative    5115
neutral     5015
positive    4870
Name: count, dtype: int64


In [ ]:
!pip install transformers torch
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Test on a sample
sample = df_sample['clean_text'].iloc[0]
tokens = tokenizer(sample, max_length=128, truncation=True, padding='max_length', return_tensors='pt')
print(tokens)

In [13]:
label_map = {'negative': 0, 'neutral': 1, 'positive': 2}
df_sample['label'] = df_sample['sentiment'].map(label_map)

In [15]:
from torch.utils.data import Dataset
import torch

class SentimentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            truncation=True,
            padding='max_length',
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'label': torch.tensor(self.labels[idx], dtype=torch.long)
        }

In [16]:
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split

texts = df_sample['clean_text'].tolist()
labels = df_sample['label'].tolist()

X_train, X_val, y_train, y_val = train_test_split(
    texts, labels, test_size=0.2, random_state=42
)

train_dataset = SentimentDataset(X_train, y_train, tokenizer)
val_dataset = SentimentDataset(X_val, y_val, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}")

Train: 12000, Val: 3000


In [17]:
from transformers import BertForSequenceClassification
from torch.optim import AdamW

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using: {device}")

model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=3
)
model = model.to(device)

optimizer = AdamW(model.parameters(), lr=2e-5)

Using: cuda


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [18]:
from torch.nn import CrossEntropyLoss

def train_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for batch in loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = outputs.logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return total_loss/len(loader), correct/total

# Train for 3 epochs
for epoch in range(3):
    loss, acc = train_epoch(model, train_loader, optimizer, device)
    print(f"Epoch {epoch+1}: Loss={loss:.4f}, Accuracy={acc:.4f}")

Epoch 1: Loss=0.7047, Accuracy=0.6774
Epoch 2: Loss=0.4694, Accuracy=0.8088
Epoch 3: Loss=0.2904, Accuracy=0.8907


In [19]:
def evaluate(model, loader, device):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = outputs.logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return correct/total

val_acc = evaluate(model, val_loader, device)
print(f"Validation Accuracy: {val_acc:.4f}")

Validation Accuracy: 0.7597


In [20]:
model.save_pretrained('sentiment_bert')
tokenizer.save_pretrained('sentiment_bert')
print("Model saved")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved


In [21]:
def predict(text, model, tokenizer, device):
    model.eval()
    encoding = tokenizer(text, max_length=128, truncation=True,
                        padding='max_length', return_tensors='pt')
    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)

    pred = outputs.logits.argmax(dim=1).item()
    labels = {0: 'negative', 1: 'neutral', 2: 'positive'}
    return labels[pred]

# Test it
print(predict("This product is absolutely amazing!", model, tokenizer, device))
print(predict("Terrible quality, waste of money", model, tokenizer, device))
print(predict("It's okay, nothing special", model, tokenizer, device))

positive
negative
neutral
